#### Context Managers

Context manager is useful to handle connections of I/O. We can create custom context manager to ensure we do not accidentally cause unexpected file conflicts or excess connections

In [26]:
class MyContextManager:
    def __init__(self, item):
        self.item = item
        self.entered = False
    
    def __enter__(self):
        print(f"Entering with {self.item}")
        self.entered = True
    
    def __exit__(self, exc_type, exc, tb):
        # if the block finishes without error, then exc_type, exc, tb will all be None
        # if there is any issues with the block, then 
        # exc_type will show the exception type
        # exc will be the actual exception
        # tb will be the traceback
        
        if self.entered and exc_type is None:
            self.entered = False
            print("Exiting")
            print(f"Entered state: {self.entered}")
        else:
            self.entered = False
            print(f"Entered state: {self.entered}")
            print(exc)
            print(tb.message)


with MyContextManager("item1") as mcm:
    print("hello")

with MyContextManager("item2") as mcm:
    1/0

Entering with item1
hello
Exiting
Entered state: False
Entering with item2
Entered state: False
division by zero


AttributeError: 'traceback' object has no attribute 'message'

### staticmethod vs classmethod

For `staticmethod`, it can be ran by itself, without taking in the class instance but for `classmethod`, the class object will be passed in and then you can work on the class instance

In [1]:
class StaticClass:
    some_attribute = "I am a static class attribute."
    
    @staticmethod
    def static_method():
        # we cannot access the class or instance attributes here, as this method does not take any parameters
        print("This is a static method.")

    @classmethod
    def class_method(cls):
        # we can interact with the class itself using the cls parameter
        print(f"{cls.some_attribute} This is a class method of {cls.__name__}.")
StaticClass.static_method()  # Output: This is a static method.
StaticClass.class_method()   # Output: This is a class method of StaticClass

This is a static method.
I am a static class attribute. This is a class method of StaticClass.


### Operator overloading

We can change how our class do operations 

In [2]:
class MyOps:
    def __init__(self, value):
        self.value = value
    def __add__(self, other):
        return self.value + other
    def __sub__(self, other):
        return self.value - other
    def __mul__(self, other):
        return self.value * other
    def __eq__(self, value):
        return self.value == value
    def __ne__(self, value):
        return self.value != value
    def __ge__(self, other):
        return self.value >= other
    def __gt__(self, other):
        return self.value > other
    def __le__(self, other):
        return self.value <= other
    def __lt__(self, other):
        return self.value < other
    
    def __int__(self):
        return int(self.value)
    def __float__(self):
        return float(self.value)
    def __str__(self):
        return self.value    

mo1 = MyOps(10)
mo2 = MyOps(20)
mo1 <= mo2
mo1 == mo2
mo1 != mo2
float(mo1)

10.0

#### metaclass
`metaclass` can be used for manupalating class creation, not instance/object creation

In order to use metaclass, we need to explicitly mention `(type)`

Using `__new__` we can initiate classes

In [3]:
class MyMetaClass(type):
    def __new__(cls, name, bases, dct, **kwargs):
        print(f"Creating class")
        print(name, bases, dct)
        dct['do_something'] = lambda self, x: print(f"{self.__class__.__name__} doing something: {x}")
        return super().__new__(cls, name, bases, dct, **kwargs)
    
class MyClass(metaclass=MyMetaClass):
    pass

MyClass().do_something("hello")

Creating class
MyClass () {'__module__': '__main__', '__qualname__': 'MyClass'}
MyClass doing something: hello


#### __call__
`__call__` will make the class object callable, basically acting like a function

In [4]:
class MyCallClass:
    def __init__(self, value):
        self.value = value
    
    def __call__(self, factor):
        print(f"Multiplying value with factor: {self.value} * {factor}")
        results = self.value * factor
        return results
    
multiply = MyCallClass(10)
multiply(3)

Multiplying value with factor: 10 * 3


30

#### __slots__
By default `__dict__` is used to store the class attributes and new attributes can be added after class is created.

`__slots__` is used to predefine how many attributes there will be in a class, so that it can reduce the storage space needed but then new attributes cannot be added and `__dict__` will not be available

In [5]:
import sys

class SlotClass:
    __slots__ = ["key", "value"]
    def __init__(self, key, value):
        self.key = key
        self.value = value
        
class DictClass:
    def __init__(self, key, value):
        self.key = key
        self.value = value

slot_class = SlotClass("my_key", "my_value")
dict_class = DictClass("my_key", "my_value")
try:
    # no longer have __dict__
    slot_class.__dict__
except Exception as e:
    print('slot_class:', e)
print("dict_class:", dict_class.__dict__)
try:
    # unable to add new attributes
    slot_class.unknown = "slot unknown"
except Exception as e:
    print('slot_class:', e)
dict_class.unknown = "dict unknown"
print('dict_class:', dict_class.unknown)
print('slot_class', slot_class.__slots__)
try:
    print('dict_class:', dict_class.__slots__)
except Exception as e:
    print('dict_class:', e)
print('slot_class', slot_class.key)
print('dict_class', dict_class.key)
print('slot_class', slot_class.value)
print('dict_class', dict_class.value)

slot_class: 'SlotClass' object has no attribute '__dict__'
dict_class: {'key': 'my_key', 'value': 'my_value'}
slot_class: 'SlotClass' object has no attribute 'unknown'
dict_class: dict unknown
slot_class ['key', 'value']
dict_class: 'DictClass' object has no attribute '__slots__'
slot_class my_key
dict_class my_key
slot_class my_value
dict_class my_value


#### Monkey Patching
Monkey patching is updating exisitng class methods with a new function instead of changing the source code itself, so that it will not affect other objects, it can also change the class method for all of the existing instances too

This should only be done if u are sure it will not affect other things unknowingly because changing methods for all instances might cause unexpected issues if there are instances that relies on that method

In [6]:
import types

class Monkey:
    def __init__(self, items):
        self.items = items
    def fetch(self, item):
        print(f"Fetching item: {item}")
        self.items.append(item)
        print(f"All items: {self.items}")

def new_fetch(self, new_item):
    print(f"New fetching new item: {new_item}")
    self.items.append(new_item)
    print(f"All items: {self.items}")

m1 = Monkey(["a", "b"])
m1.fetch("og item")
# override the original method
m1.fetch = types.MethodType(new_fetch, m1)
m1.fetch("new item")
m2 = Monkey(["c", "d"])
m2.fetch("item")
# we can override the entire method for all instances too if needed
Monkey.fetch = new_fetch # this will override the 
m2.fetch("new item")
m3 = Monkey(["f", "g"])
m3.fetch("old item gone")

Fetching item: og item
All items: ['a', 'b', 'og item']
New fetching new item: new item
All items: ['a', 'b', 'og item', 'new item']
Fetching item: item
All items: ['c', 'd', 'item']
New fetching new item: new item
All items: ['c', 'd', 'item', 'new item']
New fetching new item: old item gone
All items: ['f', 'g', 'old item gone']
